In [3]:
!pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu paddlex numpy opencv-python opencv-python-headless opencv-contrib-python

!pip install -q --no-cache-dir numpy==1.26.4
!pip install -q --no-cache-dir opencv-python-headless==4.10.0.84 --no-deps

!pip install -q --no-cache-dir olefile pillow jiwer
!pip install -q --no-cache-dir shapely pyclipper lmdb tqdm rapidfuzz attrdict

!pip install -q --no-cache-dir paddlepaddle==2.6.2 --no-deps
!pip install -q --no-cache-dir paddleocr==2.7.3 --no-deps

Found existing installation: paddleocr 2.7.3
Uninstalling paddleocr-2.7.3:
  Successfully uninstalled paddleocr-2.7.3
Found existing installation: paddlepaddle 2.6.2
Uninstalling paddlepaddle-2.6.2:
  Successfully uninstalled paddlepaddle-2.6.2
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 366.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which 

In [2]:
!pip install -q imgaug==0.4.0 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 21.3 MB/s eta 0:00:00


In [3]:
import numpy as np
print("numpy:", np.__version__)

import cv2
print("opencv:", cv2.__version__)

from paddleocr import PaddleOCR
print("PaddleOCR import OK")

numpy: 1.26.4
opencv: 4.10.0
PaddleOCR import OK


In [5]:
import os

os.makedirs("/content/hwp_files", exist_ok=True)
os.makedirs("/content/extracted_images", exist_ok=True)



In [6]:
from pathlib import Path

INPUT_DIR = Path("/content/hwp_files")

for p in INPUT_DIR.glob("*"):
    print(p.name)

인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp


In [7]:
import olefile
import zlib
from pathlib import Path

INPUT_DIR = Path("/content/hwp_files")
OUTPUT_DIR = Path("/content/extracted_images")

def detect_ext(data):

    if data.startswith(b"\xff\xd8"):
        return ".jpg"

    if data.startswith(b"\x89PNG"):
        return ".png"

    if data.startswith(b"BM"):
        return ".bmp"

    if data.startswith(b"GIF8"):
        return ".gif"

    return None


def try_decompress(data):

    for wbits in [zlib.MAX_WBITS, -zlib.MAX_WBITS, 15 + 32]:

        try:
            return zlib.decompress(data, wbits)

        except:
            pass

    return None


all_saved_files = []

for hwp_path in INPUT_DIR.glob("*.hwp"):

    print("=" * 80)
    print("파일:", hwp_path.name)

    if not olefile.isOleFile(str(hwp_path)):
        print("OLE 형식 아님")
        continue

    save_dir = OUTPUT_DIR / hwp_path.stem
    save_dir.mkdir(parents=True, exist_ok=True)

    ole = olefile.OleFileIO(str(hwp_path))

    image_count = 0

    for stream in ole.listdir():

        stream_name = "/".join(stream)

        if not stream_name.startswith("BinData/"):
            continue

        try:

            data = ole.openstream(stream).read()

            ext = detect_ext(data)

            # 압축 해제 시도
            if ext is None:

                decompressed = try_decompress(data)

                if decompressed is not None:

                    ext = detect_ext(decompressed)

                    if ext is not None:
                        data = decompressed

            # 그래도 못 찾으면 stream 확장자 사용
            if ext is None:
                ext = Path(stream_name).suffix.lower()

            if ext not in [".jpg", ".jpeg", ".png", ".bmp", ".gif"]:
                continue

            image_count += 1

            save_path = save_dir / f"img_{image_count:03d}{ext}"

            with open(save_path, "wb") as f:
                f.write(data)

            all_saved_files.append(save_path)

        except Exception as e:
            print("에러:", stream_name, e)

    ole.close()

    print("저장 이미지 수:", image_count)

print("\n전체 저장 이미지 수:", len(all_saved_files))

파일: 인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
저장 이미지 수: 5
파일: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
저장 이미지 수: 1
파일: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp
저장 이미지 수: 14
파일: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
저장 이미지 수: 5
파일: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
저장 이미지 수: 8

전체 저장 이미지 수: 33


In [8]:
from PIL import Image
import matplotlib.pyplot as plt

for img_path in all_saved_files:
    print("확인:", img_path)

    try:
        img = Image.open(img_path)
        print("열림:", img.format, img.size)

        plt.figure(figsize=(8, 5))
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")
        plt.show()

    except Exception as e:
        print("이미지로 열기 실패:", e)

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# 이미지 메타데이터/GT 생성을 위한 표준 ID 규칙
# id = source_doc_key + original_image_file_name stem
# image_file_name = id + 원본 확장자 유지

import json
from pathlib import Path

SOURCE_DOC_KEY_MAP = {
    "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보": "한영대학_학사정보",
    "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선": "한국연구재단_UICC",
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역": "한국생산기술연구원_EIP3.0",
    "인천광역시_도시계획위원회 통합관리시스템 구축용역": "인천광역시_도시계획위원회",
    "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)": "봉화군_재난통합관리",
}

SOURCE_FILE_NAME_MAP = {
    hwp_path.stem: hwp_path.name
    for hwp_path in INPUT_DIR.glob("*.hwp")
}

def make_source_doc_key(source_stem):
    if source_stem in SOURCE_DOC_KEY_MAP:
        return SOURCE_DOC_KEY_MAP[source_stem]
    return source_stem.replace(" ", "_")[:40]

def make_image_record(image_path):
    image_path = Path(image_path)
    source_stem = image_path.parent.name
    source_doc_key = make_source_doc_key(source_stem)
    original_image_file_name = image_path.name
    original_stem = image_path.stem
    ext = image_path.suffix.lower()
    image_id = f"{source_doc_key}_{original_stem}"

    return {
        "id": image_id,
        "source_file_name": SOURCE_FILE_NAME_MAP.get(source_stem, source_stem),
        "source_file_type": "hwp",
        "source_doc_key": source_doc_key,
        "original_image_file_name": original_image_file_name,
        "image_file_name": f"{image_id}{ext}",
        "image_path": str(image_path),
        "image_type": "text",      # text/table/diagram/chart/logo/low_quality 중 수동 수정
        "use_eval": True,          # 로고/장식/저품질 이미지는 False로 수정
        "gt_text": "",             # 정답 텍스트 수동 입력
        "gt_structure": {},        # 표/다이어그램 구조 정답 수동 입력
    }

image_records = [
    make_image_record(p)
    for p in sorted(OUTPUT_DIR.glob("*/*"))
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".gif"]
]

image_id_map = {
    item["id"]: item["image_path"]
    for item in image_records
}

gt_template_path = "/content/ocr_gt_template.json"
with open(gt_template_path, "w", encoding="utf-8") as f:
    json.dump(image_records, f, ensure_ascii=False, indent=2)

print("이미지 수:", len(image_records))
print("GT 템플릿 저장:", gt_template_path)
for item in image_records[:10]:
    print(item["id"], "=>", item["image_file_name"])

이미지 수: 33
GT 템플릿 저장: /content/ocr_gt_template.json
봉화군_재난통합관리_img_001 => 봉화군_재난통합관리_img_001.png
봉화군_재난통합관리_img_002 => 봉화군_재난통합관리_img_002.png
봉화군_재난통합관리_img_003 => 봉화군_재난통합관리_img_003.bmp
봉화군_재난통합관리_img_004 => 봉화군_재난통합관리_img_004.bmp
봉화군_재난통합관리_img_005 => 봉화군_재난통합관리_img_005.bmp
인천광역시_도시계획위원회_img_001 => 인천광역시_도시계획위원회_img_001.bmp
인천광역시_도시계획위원회_img_002 => 인천광역시_도시계획위원회_img_002.bmp
인천광역시_도시계획위원회_img_003 => 인천광역시_도시계획위원회_img_003.jpg
인천광역시_도시계획위원회_img_004 => 인천광역시_도시계획위원회_img_004.bmp
인천광역시_도시계획위원회_img_005 => 인천광역시_도시계획위원회_img_005.bmp


In [10]:
from paddleocr import PaddleOCR
import time

ocr = PaddleOCR(
    lang="korean",
    use_angle_cls=False,
    show_log=False,
    enable_mkldnn=False,
)

def extract_text_from_paddle_result(result):
    texts = []

    # PaddleOCR 2.x 결과는 보통:
    # [
    #   [
    #     [box, (text, score)],
    #     [box, (text, score)],
    #   ]
    # ]
    for page in result or []:
        if page is None:
            continue

        for line in page:
            try:
                text = line[1][0]
                texts.append(str(text))
            except Exception:
                pass

    return "\n".join(texts)

ocr_results = []

for item in image_records:
    image_id = item["id"]
    image_path = item["image_path"]

    print("=" * 80)
    print("OCR 실행:", image_id)

    start = time.time()
    status = "success"
    error_message = ""
    pred_text = ""

    try:
        result = ocr.ocr(image_path, cls=False)
        pred_text = extract_text_from_paddle_result(result)
    except Exception as e:
        status = "failed"
        error_message = str(e)

    latency_ms = round((time.time() - start) * 1000, 2)

    ocr_results.append({
        "id": image_id,
        "model": "paddleocr_2.7.3",
        "source_file_name": item["source_file_name"],
        "original_image_file_name": item["original_image_file_name"],
        "image_file_name": item["image_file_name"],
        "image_path": image_path,
        "pred_text": pred_text,
        "latency_ms": latency_ms,
        "status": status,
        "error_message": error_message,
    })

    print("\n[OCR RESULT]")
    print(pred_text)
    print("\nLatency:", latency_ms, "ms")
    if error_message:
        print("Error:", error_message)

download https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/Multilingual_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/ml/Multilingual_PP-OCRv3_det_infer/Multilingual_PP-OCRv3_det_infer.tar


100%|██████████| 3.85M/3.85M [00:07<00:00, 513kiB/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/korean_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/korean/korean_PP-OCRv4_rec_infer/korean_PP-OCRv4_rec_infer.tar


100%|██████████| 24.4M/24.4M [01:24<00:00, 287kiB/s] 


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:14<00:00, 150kiB/s]


OCR 실행: 봉화군_재난통합관리_img_001

[OCR RESULT]
꼬이즈인인
총리
|

Latency: 603.59 ms
OCR 실행: 봉화군_재난통합관리_img_002

[OCR RESULT]
봉화군
봉화군

Latency: 360.45 ms
OCR 실행: 봉화군_재난통합관리_img_003

[OCR RESULT]
사망실죄자
부상자
100
|
60
2
40
20
-
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021

Latency: 1922.35 ms
OCR 실행: 봉화군_재난통합관리_img_004

[OCR RESULT]
구분
2011
2012:
2013
2014
2015
2016
2017
2018
2019
2020
2021
태품
[:
태풍
호우
폭염
미산정
|은
30
3

Latency: 1808.7 ms
OCR 실행: 봉화군_재난통합관리_img_005

[OCR RESULT]
계측기
정보
자돔우람경보시셀2병
수뮈국2대
겨측데미터
뉴션
:오스
LTE
리려19개
재난상황실
마음방송막군00용개사
우린계10니
러머!!
뮤선
면계시스럼
정보수집[
사난동잡
방화
기우설금거
로게[10다
WEA/APP!
발귄/께어
데미터
미기는
시스덤연케
미존문영집
NUF
경상정보
대화
조기킴보
금경사지
하협자동차단
둔치주차장
적설개CCTVIL1O다!
DD서베
서류지
주문제에
유관기관
만
케머면돔
NDIIS
컴화도첨
홍수빔호벽
리머리!
기생치
케머면동
낙동감
홍수통제소
철한부U는
부풀시스단
위침상형전파시세신규
한국수지원공사
디지념
저닌밤승망
유션
무십경보장치
농머호
공사
시업병위
DLBES!
LTE/DME
재난빔승단말

Latency: 6645.85 ms
OCR 실행: 인천광역시_도시계획위원회_img_001

[OCR RESULT]
all
Ways
INCHEON
모든
길본
인천으로
몸한다

Latency: 725.3 ms
OCR 실행: 인천광역시_도시계획위원회_img_002

[OCR RESULT]


Latency: 50.32

In [11]:
import csv
import json

json_save_path = "/content/paddleocr_results.json"
csv_save_path = "/content/paddleocr_pred.csv"

with open(json_save_path, "w", encoding="utf-8") as f:
    json.dump(ocr_results, f, ensure_ascii=False, indent=2)

fieldnames = [
    "id",
    "model",
    "source_file_name",
    "original_image_file_name",
    "image_file_name",
    "image_path",
    "pred_text",
    "latency_ms",
    "status",
    "error_message",
]

with open(csv_save_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(ocr_results)

print("JSON 저장 완료:", json_save_path)
print("CSV 저장 완료:", csv_save_path)

JSON 저장 완료: /content/paddleocr_results.json
CSV 저장 완료: /content/paddleocr_pred.csv


In [12]:
def normalize_text(text):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

In [13]:
import json

gt_records = [
    # =========================
    # 인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp
    # =========================
    {
        "id": "인천광역시_도시계획위원회_img_001",
        "source_file_name": "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "인천광역시_도시계획위원회",
        "original_image_file_name": "img_001.bmp",
        "image_file_name": "인천광역시_도시계획위원회_img_001.bmp",
        "image_path": "/content/extracted_images/인천광역시_도시계획위원회 통합관리시스템 구축용역/img_001.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "all ways INCHEON 모든 길은 인천으로 통한다",
        "gt_structure": {
            "기관": ["인천광역시"],
            "슬로건": ["all ways INCHEON", "모든 길은 인천으로 통한다"]
        }
    },
    {
        "id": "인천광역시_도시계획위원회_img_002",
        "source_file_name": "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "인천광역시_도시계획위원회",
        "original_image_file_name": "img_002.bmp",
        "image_file_name": "인천광역시_도시계획위원회_img_002.bmp",
        "image_path": "/content/extracted_images/인천광역시_도시계획위원회 통합관리시스템 구축용역/img_002.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "",
        "gt_structure": {
            "유형": ["기관 심볼"]
        }
    },
    {
        "id": "인천광역시_도시계획위원회_img_003",
        "source_file_name": "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "인천광역시_도시계획위원회",
        "original_image_file_name": "img_003.jpg",
        "image_file_name": "인천광역시_도시계획위원회_img_003.jpg",
        "image_path": "/content/extracted_images/인천광역시_도시계획위원회 통합관리시스템 구축용역/img_003.jpg",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "인천광역시",
        "gt_structure": {
            "기관명": ["인천광역시"]
        }
    },
    {
        "id": "인천광역시_도시계획위원회_img_004",
        "source_file_name": "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "인천광역시_도시계획위원회",
        "original_image_file_name": "img_004.bmp",
        "image_file_name": "인천광역시_도시계획위원회_img_004.bmp",
        "image_path": "/content/extracted_images/인천광역시_도시계획위원회 통합관리시스템 구축용역/img_004.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "소프트웨어사업 영향평가 검토결과서 사업명 도시계획위원회 통합관리시스템 구축 용역 영향평가단계 예산편성 사업발주 소프트웨어 배포 서비스 제공 구분 1 기본정보 국가안보 유지관리 사업 소프트웨어 개발사업 데이터베이스 구축 사업 운영기관 인천광역시 예상 사용자수 내부 직원 200명 타 기관 직원 일반 국민 또는 기업 2 운영계획 3 민간 소프트웨어 시장 침해 가능성 4 사업의 필요성 공공데이터 활용 데이터 개방 Open API 제공 여부 5 종합의견 민간 소프트웨어 시장 침해 가능성 없음 인천광역시",
        "gt_structure": {
            "문서명": ["소프트웨어사업 영향평가 검토결과서"],
            "사업명": ["도시계획위원회 통합관리시스템 구축 용역"],
            "평가항목": [
                "기본정보",
                "운영계획",
                "민간 소프트웨어 시장 침해 가능성",
                "사업의 필요성",
                "종합의견"
            ],
            "운영기관": ["인천광역시"],
            "예상 사용자수": {
                "내부 직원": "200명",
                "타 기관 직원": "",
                "일반 국민 또는 기업": ""
            },
            "종합의견": ["민간 소프트웨어 시장 침해 가능성 없음"]
        }
    },
    {
        "id": "인천광역시_도시계획위원회_img_005",
        "source_file_name": "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "인천광역시_도시계획위원회",
        "original_image_file_name": "img_005.bmp",
        "image_file_name": "인천광역시_도시계획위원회_img_005.bmp",
        "image_path": "/content/extracted_images/인천광역시_도시계획위원회 통합관리시스템 구축용역/img_005.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "소프트웨어 개발사업의 적정 사업기간 종합 산정서 도시계획위원회 통합관리시스템 구축 용역 합계 월 6개월 기능점수 기반 적정 사업기간 산정 사업규모 기능점수 기능점수 FP 기준 보정 개발기간 사업기간계수 정보공개 요구사항 분석 설계 구현 시험 사업기초자료 사업계획서 산출내역서 유사사업 자료 조달청 유사사업에 따라 산정 기간 비교 결과 적정한 것으로 판단됨 기타 특이사항 개발 후 유지 운영되기 전 안정화기간이 필요할 것으로 예상되므로 7개월이 필요한 것으로 사료됨 종합의견 적정 사업기간 6개월 소프트웨어사업 계약 및 관리감독에 관한 지침 제10조제3항에 따른 소프트웨어 개발사업의 적정 사업기간을 위와 같이 산정합니다 2024년 2월 5일 인천광역시장 귀하",
        "gt_structure": {
            "문서명": ["소프트웨어 개발사업의 적정 사업기간 종합 산정서"],
            "사업명": ["도시계획위원회 통합관리시스템 구축 용역"],
            "검토항목": [
                "기능점수 기반 적정 사업기간 산정",
                "사업기초자료",
                "유사사업 자료",
                "기타 특이사항",
                "종합의견"
            ],
            "적정 사업기간": ["6개월"],
            "기타 특이사항": ["안정화기간 필요", "7개월 필요 의견"],
            "작성일": ["2024년 2월 5일"],
            "수신": ["인천광역시장 귀하"]
        }
    },

    # =========================
    # 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
    # =========================
    {
        "id": "봉화군_재난통합관리_img_001",
        "source_file_name": "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
        "source_file_type": "hwp",
        "source_doc_key": "봉화군_재난통합관리",
        "original_image_file_name": "img_001.png",
        "image_file_name": "봉화군_재난통합관리_img_001.png",
        "image_path": "/content/extracted_images/경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)/img_001.png",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "오래된 준비된 희망찬 봉화",
        "gt_structure": {
            "슬로건": ["오래된", "준비된", "희망찬 봉화"]
        }
    },
    {
        "id": "봉화군_재난통합관리_img_002",
        "source_file_name": "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
        "source_file_type": "hwp",
        "source_doc_key": "봉화군_재난통합관리",
        "original_image_file_name": "img_002.png",
        "image_file_name": "봉화군_재난통합관리_img_002.png",
        "image_path": "/content/extracted_images/경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)/img_002.png",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "봉화군",
        "gt_structure": {
            "기관명": ["봉화군"]
        }
    },
    {
        "id": "봉화군_재난통합관리_img_003",
        "source_file_name": "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
        "source_file_type": "hwp",
        "source_doc_key": "봉화군_재난통합관리",
        "original_image_file_name": "img_003.bmp",
        "image_file_name": "봉화군_재난통합관리_img_003.bmp",
        "image_path": "/content/extracted_images/경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)/img_003.bmp",
        "image_type": "chart",
        "use_eval": True,
        "gt_text": "사망(실종)자 부상자 2011 78 60 2012 16 37 2013 4 0 2014 2 0 2015 0 0 2016 7 4 2017 7 15 2018 53 4 2019 48 37 2020 75 28 2021 42 42",
        "gt_structure": {
            "컬럼": ["2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021"],
            "통계": {
                "사망(실종)자": {
                    "2011": 78, "2012": 16, "2013": 4, "2014": 2, "2015": 0, "2016": 7,
                    "2017": 7, "2018": 53, "2019": 48, "2020": 75, "2021": 42
                },
                "부상자": {
                    "2011": 60, "2012": 37, "2013": 0, "2014": 0, "2015": 0, "2016": 4,
                    "2017": 15, "2018": 4, "2019": 37, "2020": 28, "2021": 42
                }
            }
        }
    },
    {
        "id": "봉화군_재난통합관리_img_004",
        "source_file_name": "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
        "source_file_type": "hwp",
        "source_doc_key": "봉화군_재난통합관리",
        "original_image_file_name": "img_004.bmp",
        "image_file_name": "봉화군_재난통합관리_img_004.bmp",
        "image_path": "/content/extracted_images/경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)/img_004.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "구분 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 태풍 1 14 - - - 6 - 2 18 2 - 호우 77 2 4 2 - 1 7 2 - 44 3 태풍호우 - - - - - - - 1 - - - 폭염 미산정 48 30 29 39",
        "gt_structure": {
            "컬럼": ["2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021"],
            "통계": {
                "태풍": {"2011": 1, "2012": 14, "2013": "-", "2014": "-", "2015": "-", "2016": 6, "2017": "-", "2018": 2, "2019": 18, "2020": 2, "2021": "-"},
                "호우": {"2011": 77, "2012": 2, "2013": 4, "2014": 2, "2015": "-", "2016": 1, "2017": 7, "2018": 2, "2019": "-", "2020": 44, "2021": 3},
                "태풍호우": {"2011": "-", "2012": "-", "2013": "-", "2014": "-", "2015": "-", "2016": "-", "2017": "-", "2018": 1, "2019": "-", "2020": "-", "2021": "-"},
                "폭염": {"2011": "미산정", "2012": "미산정", "2013": "미산정", "2014": "미산정", "2015": "미산정", "2016": "미산정", "2017": "미산정", "2018": 48, "2019": 30, "2020": 29, "2021": 39}
            }
        }
    },
    {
        "id": "봉화군_재난통합관리_img_005",
        "source_file_name": "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
        "source_file_type": "hwp",
        "source_doc_key": "봉화군_재난통합관리",
        "original_image_file_name": "img_005.bmp",
        "image_file_name": "봉화군_재난통합관리_img_005.bmp",
        "image_path": "/content/extracted_images/경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)/img_005.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "계측기 정보 수위국(2대) LTE 우량국(10개) LTE 조기경보 수위계 강우량계(10대) 적설계(10대) 강우설량계 로거(10대) 영상정보 적설계 CCTV(10대) 유관기관 연동 NDMS 경북도청 기상청 낙동강 홍수통제소 한국수자원공사 농어촌 공사 봉화군 재난상황실 행정망 정보수집 연계시스템 NVR 기존운용 중 제어망 재난통합 관리시스템 WEB APP DB 서버 방화벽 VPN 방화벽 모바일 자동우량경보시설 유선 LTE 마을방송 유선 LTE 이기종 시스템 연계 조기경보 급경사지 하천자동차단 둔치주차장 저류지 수문제어 제어 연동 홍수방호벽 제어 제어 연동 위험상황전파시설 신규 유선 무선경보장치 LTE DMB 재난방송단말 디지털 재난방송 DMB EWS 사업범위",
        "gt_structure": {
            "계측기 정보": ["수위국(2대)", "우량국(10개)", "조기경보 수위계", "강우량계(10대)", "적설계(10대)", "강우설량계 로거(10대)"],
            "영상정보": ["적설계 CCTV(10대)"],
            "유관기관 연동": ["NDMS", "경북도청", "기상청", "낙동강 홍수통제소", "한국수자원공사", "농어촌 공사"],
            "봉화군 재난상황실": {
                "행정망": ["정보수집 연계시스템", "NVR(기존운용 중)"],
                "제어망": ["재난통합 관리시스템(WEB/APP)", "DB 서버"],
                "보안": ["방화벽", "VPN 방화벽"]
            },
            "연계 시스템": {
                "자동우량경보시설": ["유선", "LTE"],
                "마을방송": ["유선", "LTE"],
                "이기종 시스템 연계": ["조기경보", "급경사지", "하천자동차단", "둔치주차장"],
                "저류지 수문제어": ["제어 연동"],
                "홍수방호벽 제어": ["제어 연동"],
                "위험상황전파시설": ["신규", "유선", "무선경보장치", "LTE/DMB", "재난방송단말"],
                "디지털 재난방송": ["DMB(EWS)"]
            },
            "사용자 채널": ["모바일"]
        }
    },
        {
        "id": "한영대학_학사정보_img_001",
        "source_file_name": "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한영대학_학사정보",
        "original_image_file_name": "img_001.jpg",
        "image_file_name": "한영대학_학사정보_img_001.jpg",
        "image_path": "/content/extracted_images/한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보/img_001.jpg",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "한영대학교 HANYEONG UNIVERSITY HANYEONG UNIVERSITY YEOSU 1993 KOREA",
        "gt_structure": {
            "기관명": ["한영대학교"],
            "영문명": ["HANYEONG UNIVERSITY"],
            "로고문구": ["HANYEONG UNIVERSITY", "YEOSU 1993 KOREA"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_001",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_001.bmp",
        "image_file_name": "한국연구재단_UICC_img_001.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_001.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "SW개발사업의 적정 사업기간 종합 검토서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 항목별 검토 의견 검토항목 검토의견 추정 사업기간 기능점수(FP) 기반 SW사업 적정 개발기간 산정 사업기초자료 유사사업 자료 기타 특이사항 해당없음 종합의견 적정 사업기간 7개월 2024년 9월 11일 한국연구재단 귀하",
        "gt_structure": {
            "문서명": ["SW개발사업의 적정 사업기간 종합 검토서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "검토항목": ["기능점수(FP) 기반 SW사업 적정 개발기간 산정", "사업기초자료", "유사사업 자료", "기타 특이사항", "종합의견"],
            "적정 사업기간": ["7개월"],
            "작성일": ["2024년 9월 11일"],
            "수신": ["한국연구재단 귀하"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_002",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_002.bmp",
        "image_file_name": "한국연구재단_UICC_img_002.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_002.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "SW개발사업의 적정 사업기간 종합 검토서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 항목별 검토 의견 기능점수(FP) 기반 SW사업 적정 개발기간 산정 사업기초자료 유사사업 자료 기타 특이사항 해당없음 종합의견 적정 사업기간 7개월 2024년 9월 12일 한국연구재단 귀하",
        "gt_structure": {
            "문서명": ["SW개발사업의 적정 사업기간 종합 검토서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "검토항목": ["기능점수(FP) 기반 SW사업 적정 개발기간 산정", "사업기초자료", "유사사업 자료", "기타 특이사항", "종합의견"],
            "적정 사업기간": ["7개월"],
            "작성일": ["2024년 9월 12일"],
            "수신": ["한국연구재단 귀하"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_003",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_003.bmp",
        "image_file_name": "한국연구재단_UICC_img_003.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_003.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "SW개발사업의 적정 사업기간 종합 검토서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 항목별 검토 의견 UICC 기능개선 사업의 기능점수와 사업 범위에 따른 적정 개발기간 검토 사업기초자료 유사사업 자료 기타 특이사항 해당사항 없음 종합의견 적정 사업기간 7개월 2024년 9월 11일 한국연구재단 귀하",
        "gt_structure": {
            "문서명": ["SW개발사업의 적정 사업기간 종합 검토서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "검토항목": ["기능점수(FP) 기반 SW사업 적정 개발기간 산정", "사업기초자료", "유사사업 자료", "기타 특이사항", "종합의견"],
            "적정 사업기간": ["7개월"],
            "작성일": ["2024년 9월 11일"],
            "수신": ["한국연구재단 귀하"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_004",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_004.bmp",
        "image_file_name": "한국연구재단_UICC_img_004.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_004.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "SW개발사업의 적정 사업기간 종합 검토서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 항목별 검토 의견 기능점수(FP) 기반 SW사업 적정 개발기간 산정 사업기초자료 유사사업 자료 기타 특이사항 해당사항 없음 종합의견 적정 사업기간 7개월 2024년 9월 10일 한국연구재단 귀하",
        "gt_structure": {
            "문서명": ["SW개발사업의 적정 사업기간 종합 검토서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "검토항목": ["기능점수(FP) 기반 SW사업 적정 개발기간 산정", "사업기초자료", "유사사업 자료", "기타 특이사항", "종합의견"],
            "적정 사업기간": ["7개월"],
            "작성일": ["2024년 9월 10일"],
            "수신": ["한국연구재단 귀하"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_005",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_005.bmp",
        "image_file_name": "한국연구재단_UICC_img_005.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_005.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "SW개발사업의 적정 사업기간 종합 검토서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 항목별 검토 의견 기능점수(FP) 기반 SW사업 적정 개발기간 산정 사업기초자료 유사사업 자료 기타 특이사항 없음 종합의견 적정 사업기간 7개월 2024년 9월 10일 한국연구재단 귀하",
        "gt_structure": {
            "문서명": ["SW개발사업의 적정 사업기간 종합 검토서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "검토항목": ["기능점수(FP) 기반 SW사업 적정 개발기간 산정", "사업기초자료", "유사사업 자료", "기타 특이사항", "종합의견"],
            "적정 사업기간": ["7개월"],
            "작성일": ["2024년 9월 10일"],
            "수신": ["한국연구재단 귀하"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_006",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_006.bmp",
        "image_file_name": "한국연구재단_UICC_img_006.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_006.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "소프트웨어사업 영향평가 검토결과서 사업명 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업 영향평가단계 예산편성 사업발주 소프트웨어 배포 서비스 제공 주요 내용 기본정보 운영계획 민간 소프트웨어 시장 침해 가능성 사업의 필요성 공공성 검토 종합의견 민간 소프트웨어 시장 침해 가능성 없음 한국연구재단",
        "gt_structure": {
            "문서명": ["소프트웨어사업 영향평가 검토결과서"],
            "사업명": ["2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선 사업"],
            "평가항목": ["기본정보", "운영계획", "민간 소프트웨어 시장 침해 가능성", "사업의 필요성", "종합의견"],
            "기관명": ["한국연구재단"],
            "종합의견": ["민간 소프트웨어 시장 침해 가능성 없음"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_007",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_007.bmp",
        "image_file_name": "한국연구재단_UICC_img_007.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_007.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_008",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_008.jpg",
        "image_file_name": "한국연구재단_UICC_img_008.jpg",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_008.jpg",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NATIONAL RESEARCH FOUNDATION OF KOREA NRF 2009 한국연구재단",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "영문명": ["NATIONAL RESEARCH FOUNDATION OF KOREA"],
            "약어": ["NRF"],
            "연도": ["2009"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_009",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_009.jpg",
        "image_file_name": "한국연구재단_UICC_img_009.jpg",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_009.jpg",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단 National Research Foundation of Korea",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "영문명": ["National Research Foundation of Korea"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_010",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_010.jpg",
        "image_file_name": "한국연구재단_UICC_img_010.jpg",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_010.jpg",
        "image_type": "decorative",
        "use_eval": False,
        "gt_text": "",
        "gt_structure": {
            "유형": ["폴더 아이콘", "장식 이미지"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_011",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_011.bmp",
        "image_file_name": "한국연구재단_UICC_img_011.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_011.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_012",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_012.bmp",
        "image_file_name": "한국연구재단_UICC_img_012.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_012.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_013",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_013.bmp",
        "image_file_name": "한국연구재단_UICC_img_013.bmp",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_013.bmp",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국연구재단_UICC_img_014",
        "source_file_name": "한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국연구재단_UICC",
        "original_image_file_name": "img_014.jpg",
        "image_file_name": "한국연구재단_UICC_img_014.jpg",
        "image_path": "/content/extracted_images/한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선/img_014.jpg",
        "image_type": "logo",
        "use_eval": False,
        "gt_text": "NRF 한국연구재단 National Research Foundation of Korea",
        "gt_structure": {
            "기관명": ["한국연구재단"],
            "영문명": ["National Research Foundation of Korea"],
            "약어": ["NRF"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_001",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_001.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_001.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_001.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "EIP 시스템 구성현황 한국생산기술연구원 통합정보시스템 업무포털(EIP) 외부 홈페이지 SNS 기관지원시스템 연구관리시스템 지능형 포털 경영정보시스템 전자결재시스템 행정정보시스템 업무프로세스관리 협업 지원 업무 지원 사용자 일반 보직자 연구책임자 관리자 연구원 실무자 원격접속 출연 근무자 모바일 사용자",
        "gt_structure": {
            "문서명": ["EIP 시스템 구성현황"],
            "시스템명": ["한국생산기술연구원 통합정보시스템 업무포털(EIP)"],
            "외부": ["중소기업", "대학", "연구기관", "정부기관", "국가과학기술정보서비스", "기타"],
            "주요 시스템": ["기관지원시스템", "연구관리시스템", "경영정보시스템", "전자결재시스템", "행정정보시스템", "업무프로세스관리", "협업 지원", "업무 지원"],
            "사용자": ["일반", "보직자", "연구책임자", "관리자", "연구원", "실무자", "출연 근무자", "모바일 사용자"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_002",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_002.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_002.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_002.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "EIP 시스템 아키텍처(소프트웨어) DBMS WEB Application Server Server FrameWork UI FrameWork 포털 및 검색 추천 연계 시스템 Informix IBM WebSphere proworks5 WebSquare5 Marine4 Dplatform IPMS Astele ELN ProQ G2B Tibero Framework Common Service Business Component Reporting Tools Oz Report StreamDocs 통합인증 및 보안 KSign Flow control Git GitLab ChangeFlow-SR CRM 검토 및 테스트",
        "gt_structure": {
            "문서명": ["EIP 시스템 아키텍처(소프트웨어)"],
            "영역": ["DBMS", "WEB Application Server", "Server FrameWork", "UI FrameWork", "포털 및 검색 추천", "연계 시스템"],
            "소프트웨어": ["Informix", "IBM WebSphere", "proworks5", "WebSquare5", "Marine4", "Dplatform", "IPMS", "Astele ELN", "ProQ G2B", "Tibero", "OZ Report", "StreamDocs", "KSign", "Flow control"],
            "관리도구": ["Git", "GitLab", "ChangeFlow-SR", "CRM", "검토 및 테스트"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_003",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_003.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_003.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_003.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "고압가스 안전관리시스템 Process Map (24.2.13.) Start 고압가스 내역비계 신규업체 등록 고압가스 DB 제품규격 및 신고대상 저장목록 확인 고압가스 사용업체관리 검토 구입신청 검사 사용 및 변경 구매목록 수령확인 완료 보유 DB 확보 신고 및 허가대상 확인 사전안전성 정보 DB 신규 고압가스 제품 DB 등록 구입검토 검사 공급업체 회수 완료 폐기완료 물질정보포털(MSDS 등)",
        "gt_structure": {
            "문서명": ["고압가스 안전관리시스템 Process Map"],
            "기준일": ["24.2.13."],
            "프로세스": ["신규업체 등록", "고압가스 DB 확인", "고압가스 사용업체관리 검토", "구입신청", "검사", "사용 및 변경 구매목록", "수령확인", "공급업체 회수 완료", "폐기완료"],
            "데이터": ["사전안전성 정보 DB", "신규 고압가스 제품 DB 등록", "물질정보포털(MSDS 등)"],
            "결정": ["구입검토"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_004",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_004.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_004.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_004.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "제품출입 확인등록 단계 연구실책임자 안전보건총괄실 안전보건팀 구매자산실 공급업체 OUTPUT 활동사항 START 고압가스 내역비계 신규업체 등록 신고대상 제품규격 확인 보유 DB 확보 신규제품 제품 등록 공급업체 제품 확인 신규제품 품의 신청 위험성 검토서 작성 MSDS 물질정보포털 담당자확인 사용자 신청",
        "gt_structure": {
            "단계": ["제품출입 확인등록 단계"],
            "역할": ["연구실책임자", "안전보건총괄실", "안전보건팀", "구매자산실", "공급업체"],
            "프로세스": ["고압가스 내역비계 신규업체 등록", "신고대상 제품규격 확인", "보유 DB 확보", "신규제품 제품 등록", "공급업체 제품 확인", "신규제품 품의 신청", "위험성 검토서 작성"],
            "산출물": ["MSDS", "물질정보포털", "담당자확인", "사용자 신청"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_005",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_005.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_005.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_005.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "사전안전성검토 단계 구매 단계 제품 입고 알림 및 사전안전성검토 대상여부 확인 구매요청 품목 제품 구매요청 신청 안전검토 확인 구매요구서 검토 발주요청 접수 승인 제품입고요청 작성 및 입고 확인 검사 검수",
        "gt_structure": {
            "단계": ["사전안전성검토 단계", "구매 단계"],
            "프로세스": ["제품 입고 알림", "사전안전성검토 대상여부 확인", "구매요청 품목", "제품 구매요청 신청", "안전검토 확인", "구매요구서 검토", "발주요청", "접수", "승인", "제품입고요청 작성", "입고 확인", "검사", "검수"],
            "결정": ["구매요구서 검토", "승인"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_006",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_006.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_006.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_006.bmp",
        "image_type": "diagram",
        "use_eval": True,
        "gt_text": "취급 단계 사용저장 폐기 단계 입고 수령상품명 확인 사용 및 잔량관리 수정 현장근무자 확인 안전관리 대장 확인 수령관리 폐기 신청 및 접수 승인 폐기 신청 및 회수신청 확인 보유 DB 정리 폐기 DB 현황 확인 폐기 완료",
        "gt_structure": {
            "단계": ["취급 단계(사용저장)", "폐기 단계(반납)"],
            "프로세스": ["입고 수령상품명 확인", "사용 및 잔량관리 수정", "현장근무자 확인", "안전관리 대장 확인", "수령관리", "폐기 신청 및 접수 승인", "폐기 신청 및 회수신청 확인", "보유 DB 정리", "폐기 DB 현황 확인", "폐기 완료"],
            "산출물": ["안전관리 대장", "보유 DB", "폐기 DB"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_007",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_007.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_007.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_007.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "소프트웨어사업 영향평가 검토결과서 사업명 EIP3.0 고압가스 안전관리 시스템 구축 용역 영향평가단계 예산편성 사업발주 소프트웨어 배포 서비스 제공 주요내용 EIP3.0 고압가스 화학물질 안전관리 시스템 구축 사업기간 2024년 9월 2024년 12월 기본정보 운영기관 한국생산기술연구원 예상 사용자수 내부 직원 2,300명 민간 소프트웨어 시장 침해 가능성 주요 기능 동일 유사한 민간 소프트웨어",
        "gt_structure": {
            "문서명": ["소프트웨어사업 영향평가 검토결과서"],
            "사업명": ["EIP3.0 고압가스 안전관리 시스템 구축 용역"],
            "주요내용": ["EIP3.0 고압가스 화학물질 안전관리 시스템 구축"],
            "사업기간": ["2024년 9월", "2024년 12월"],
            "운영기관": ["한국생산기술연구원"],
            "예상 사용자수": {"내부 직원": "2,300명"},
            "평가항목": ["기본정보", "운영계획", "민간 소프트웨어 시장 침해 가능성"]
        }
    },
    {
        "id": "한국생산기술연구원_EIP3.0_img_008",
        "source_file_name": "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
        "source_file_type": "hwp",
        "source_doc_key": "한국생산기술연구원_EIP3.0",
        "original_image_file_name": "img_008.bmp",
        "image_file_name": "한국생산기술연구원_EIP3.0_img_008.bmp",
        "image_path": "/content/extracted_images/한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역/img_008.bmp",
        "image_type": "table",
        "use_eval": True,
        "gt_text": "소프트웨어사업 영향평가 검토결과서 사업의 필요성 공공성 검토 법령에 규정된 사업 공공데이터 활용 공공서비스 제공 및 장비 가이드라인 준수 사업을 통한 민간 소프트웨어 시장 활성화 기여 Open API 등을 통한 데이터 개방 민간 소프트웨어 시장 침해 가능성 없음 종합의견 민간 소프트웨어 시장 침해 가능성을 최소화하여 사업 추진 2024년 8월 19일 한국생산기술연구원",
        "gt_structure": {
            "문서명": ["소프트웨어사업 영향평가 검토결과서"],
            "평가항목": ["사업의 필요성", "공공성 검토", "종합의견"],
            "검토내용": ["법령에 규정된 사업", "공공데이터 활용", "공공서비스 제공", "Open API", "데이터 개방"],
            "종합의견": ["민간 소프트웨어 시장 침해 가능성 없음", "민간 소프트웨어 시장 침해 가능성을 최소화하여 사업 추진"],
            "작성일": ["2024년 8월 19일"],
            "기관명": ["한국생산기술연구원"]
        }
    }
]

gt_path = "/content/ocr_gt.json"
with open(gt_path, "w", encoding="utf-8") as f:
    json.dump(gt_records, f, ensure_ascii=False, indent=2)


for item in gt_records:
    if item["image_type"] in ["logo"]:
        item["use_eval"] = False
    else:
        item["use_eval"] = True

print("전체 GT 수:", len(gt_records))
print("평가 대상 수:", sum(1 for item in gt_records if item.get("use_eval")))
print("평가 대상:")
for item in gt_records:
    if item.get("use_eval"):
        print("-", item["id"], item["image_type"])

전체 GT 수: 33
평가 대상 수: 20
평가 대상:
- 인천광역시_도시계획위원회_img_004 table
- 인천광역시_도시계획위원회_img_005 table
- 봉화군_재난통합관리_img_003 chart
- 봉화군_재난통합관리_img_004 table
- 봉화군_재난통합관리_img_005 diagram
- 한국연구재단_UICC_img_001 table
- 한국연구재단_UICC_img_002 table
- 한국연구재단_UICC_img_003 table
- 한국연구재단_UICC_img_004 table
- 한국연구재단_UICC_img_005 table
- 한국연구재단_UICC_img_006 table
- 한국연구재단_UICC_img_010 decorative
- 한국생산기술연구원_EIP3.0_img_001 diagram
- 한국생산기술연구원_EIP3.0_img_002 diagram
- 한국생산기술연구원_EIP3.0_img_003 diagram
- 한국생산기술연구원_EIP3.0_img_004 diagram
- 한국생산기술연구원_EIP3.0_img_005 diagram
- 한국생산기술연구원_EIP3.0_img_006 diagram
- 한국생산기술연구원_EIP3.0_img_007 table
- 한국생산기술연구원_EIP3.0_img_008 table


In [14]:
from jiwer import cer, wer
from difflib import SequenceMatcher
import csv
import json
import re

def normalize_text(text):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compact_text(text):
    text = normalize_text(text)
    text = re.sub(r"[^0-9a-z가-힣]+", "", text)
    return text

def flatten_structure(obj, path=""):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            next_path = f"{path}.{key}" if path else str(key)
            rows.extend(flatten_structure(value, next_path))

    elif isinstance(obj, list):
        for value in obj:
            rows.extend(flatten_structure(value, path))

    else:
        value = str(obj).strip()
        if value:
            rows.append({
                "field_path": path,
                "expected_value": value
            })

    return rows

def value_match_score(expected, pred_text):
    expected_norm = compact_text(expected)
    pred_norm = compact_text(pred_text)

    if not expected_norm:
        return 0, False

    if expected_norm in pred_norm:
        return 1.0, True

    score = SequenceMatcher(None, expected_norm, pred_norm).ratio()
    return score, score >= 0.65

pred_by_id = {
    item["id"]: item
    for item in ocr_results
}

eval_rows = []
field_rows = []

for gt_item in gt_records:
    if not gt_item.get("use_eval", True):
        continue

    image_id = gt_item["id"]
    pred_item = pred_by_id.get(image_id)

    gt_text = normalize_text(gt_item.get("gt_text", ""))
    pred_text = normalize_text(pred_item.get("pred_text", "") if pred_item else "")

    cer_score = cer(gt_text, pred_text) if gt_text or pred_text else 0
    wer_score = wer(gt_text, pred_text) if gt_text or pred_text else 0
    text_similarity = SequenceMatcher(None, gt_text, pred_text).ratio()

    expected_fields = flatten_structure(gt_item.get("gt_structure", {}))
    matched_count = 0

    for field in expected_fields:
        score, matched = value_match_score(field["expected_value"], pred_text)
        if matched:
            matched_count += 1

        field_rows.append({
            "id": image_id,
            "image_type": gt_item.get("image_type", ""),
            "field_path": field["field_path"],
            "expected_value": field["expected_value"],
            "match_score": score,
            "matched": matched,
        })

    field_hit_rate = matched_count / len(expected_fields) if expected_fields else None

    row = {
        "id": image_id,
        "image_type": gt_item.get("image_type", ""),
        "status": pred_item.get("status", "missing") if pred_item else "missing",
        "cer": cer_score,
        "wer": wer_score,
        "text_similarity": text_similarity,
        "field_hit_rate": field_hit_rate,
        "matched_fields": matched_count,
        "total_fields": len(expected_fields),
        "latency_ms": pred_item.get("latency_ms") if pred_item else None,
    }
    eval_rows.append(row)

    print("=" * 80)
    print(image_id)
    print("type:", row["image_type"])
    print("status:", row["status"])
    print("문자 유사도:", round(text_similarity * 100, 2), "%")
    print("CER:", round(cer_score, 4))
    print("WER:", round(wer_score, 4))
    if field_hit_rate is not None:
        print("필드값 매칭률:", round(field_hit_rate * 100, 2), "%", f"({matched_count}/{len(expected_fields)})")
    print("latency_ms:", row["latency_ms"])

if eval_rows:
    avg_similarity = sum(r["text_similarity"] for r in eval_rows) / len(eval_rows)

    field_rates = [
        r["field_hit_rate"]
        for r in eval_rows
        if r["field_hit_rate"] is not None
    ]
    avg_field_hit_rate = sum(field_rates) / len(field_rates) if field_rates else None

    fail_rate = sum(1 for r in eval_rows if r["status"] != "success") / len(eval_rows)

    print("\n[SUMMARY]")
    print("평가 이미지 수:", len(eval_rows))
    print("평균 문자 유사도:", round(avg_similarity * 100, 2), "%")
    if avg_field_hit_rate is not None:
        print("평균 필드값 매칭률:", round(avg_field_hit_rate * 100, 2), "%")
    print("실패율:", round(fail_rate * 100, 2), "%")

with open("/content/ocr_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(eval_rows, f, ensure_ascii=False, indent=2)

with open("/content/ocr_field_match.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "image_type", "field_path", "expected_value", "match_score", "matched"]
    )
    writer.writeheader()
    writer.writerows(field_rows)



인천광역시_도시계획위원회_img_004
type: table
status: success
문자 유사도: 30.85 %
CER: 2.3669
WER: 2.7059
필드값 매칭률: 30.0 % (3/10)
latency_ms: 19232.82
인천광역시_도시계획위원회_img_005
type: table
status: success
문자 유사도: 27.82 %
CER: 0.8125
WER: 1.093
필드값 매칭률: 50.0 % (6/12)
latency_ms: 13002.15
봉화군_재난통합관리_img_003
type: chart
status: success
문자 유사도: 61.46 %
CER: 0.4918
WER: 0.7429
필드값 매칭률: 78.79 % (26/33)
latency_ms: 1922.35
봉화군_재난통합관리_img_004
type: table
status: success
문자 유사도: 62.81 %
CER: 0.5159
WER: 0.7222
필드값 매칭률: 60.0 % (33/55)
latency_ms: 1808.7
봉화군_재난통합관리_img_005
type: diagram
status: success
문자 유사도: 25.31 %
CER: 0.8356
WER: 0.974
필드값 매칭률: 11.11 % (4/36)
latency_ms: 6645.85
한국연구재단_UICC_img_001
type: table
status: success
문자 유사도: 37.83 %
CER: 2.709
WER: 2.878
필드값 매칭률: 70.0 % (7/10)
latency_ms: 14022.4
한국연구재단_UICC_img_002
type: table
status: success
문자 유사도: 33.05 %
CER: 3.5673
WER: 4.2162
필드값 매칭률: 70.0 % (7/10)
latency_ms: 16155.44
한국연구재단_UICC_img_003
type: table
status: success
문자 유사도: 25.31 %
CER: 2.4385
WE

In [15]:
from difflib import SequenceMatcher
import json
import re

def normalize_text(text):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compact_text(text):
    text = normalize_text(text)
    text = re.sub(r"[^0-9a-z가-힣]+", "", text)
    return text

def is_value_matched(expected, pred_text, threshold=0.65):
    expected_norm = compact_text(expected)
    pred_norm = compact_text(pred_text)

    if not expected_norm:
        return False, 0

    if expected_norm in pred_norm:
        return True, 1.0

    score = SequenceMatcher(None, expected_norm, pred_norm).ratio()
    return score >= threshold, score

def build_pred_structure_from_gt_schema(gt_obj, pred_text, threshold=0.65):
    """
    OCR-only 구조화:
    - gt_structure의 구조를 스키마처럼 사용
    - OCR 결과에 매칭되는 값만 남김
    - 매칭 안 되는 값은 빈 리스트/빈 dict로 둠
    """

    if isinstance(gt_obj, dict):
        result = {}
        for key, value in gt_obj.items():
            result[key] = build_pred_structure_from_gt_schema(
                value,
                pred_text,
                threshold=threshold
            )
        return result

    if isinstance(gt_obj, list):
        matched_values = []
        for value in gt_obj:
            matched, score = is_value_matched(value, pred_text, threshold=threshold)
            if matched:
                matched_values.append(str(value))
        return matched_values

    value = str(gt_obj).strip()
    matched, score = is_value_matched(value, pred_text, threshold=threshold)
    return value if matched else ""

pred_by_id = {
    item["id"]: item
    for item in ocr_results
}

pred_structure_records = []

for gt_item in gt_records:
    image_id = gt_item["id"]
    pred_item = pred_by_id.get(image_id)

    pred_text = pred_item.get("pred_text", "") if pred_item else ""

    pred_structure = build_pred_structure_from_gt_schema(
        gt_item.get("gt_structure", {}),
        pred_text,
        threshold=0.65
    )

    record = {
        "id": image_id,
        "source_file_name": gt_item.get("source_file_name", ""),
        "source_file_type": gt_item.get("source_file_type", ""),
        "source_doc_key": gt_item.get("source_doc_key", ""),
        "original_image_file_name": gt_item.get("original_image_file_name", ""),
        "image_file_name": gt_item.get("image_file_name", ""),
        "image_path": gt_item.get("image_path", ""),
        "image_type": gt_item.get("image_type", ""),
        "use_eval": gt_item.get("use_eval", True),
        "pred_text": pred_text,
        "pred_structure": pred_structure,
    }

    pred_structure_records.append(record)

# 보기 좋게 출력
for record in pred_structure_records:
    if not record.get("use_eval"):
        continue

    print("=" * 100)
    print("id:", record["id"])
    print("image_type:", record["image_type"])
    print("[PRED_STRUCTURE]")
    print(json.dumps(record["pred_structure"], ensure_ascii=False, indent=2))
    print()

save_path = "/content/paddleocr_pred_structure.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(pred_structure_records, f, ensure_ascii=False, indent=2)

print("저장 완료:", save_path)

id: 인천광역시_도시계획위원회_img_004
image_type: table
[PRED_STRUCTURE]
{
  "문서명": [
    "소프트웨어사업 영향평가 검토결과서"
  ],
  "사업명": [],
  "평가항목": [
    "기본정보",
    "종합의견"
  ],
  "운영기관": [],
  "예상 사용자수": {
    "내부 직원": "",
    "타 기관 직원": "",
    "일반 국민 또는 기업": ""
  },
  "종합의견": []
}

id: 인천광역시_도시계획위원회_img_005
image_type: table
[PRED_STRUCTURE]
{
  "문서명": [
    "소프트웨어 개발사업의 적정 사업기간 종합 산정서"
  ],
  "사업명": [],
  "검토항목": [
    "사업기초자료",
    "유사사업 자료",
    "종합의견"
  ],
  "적정 사업기간": [
    "6개월"
  ],
  "기타 특이사항": [],
  "작성일": [],
  "수신": [
    "인천광역시장 귀하"
  ]
}

id: 봉화군_재난통합관리_img_003
image_type: chart
[PRED_STRUCTURE]
{
  "컬럼": [
    "2011",
    "2012",
    "2013",
    "2014",
    "2015",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021"
  ],
  "통계": {
    "사망(실종)자": {
      "2011": "",
      "2012": "16",
      "2013": "4",
      "2014": "2",
      "2015": "0",
      "2016": "7",
      "2017": "7",
      "2018": "",
      "2019": "",
      "2020": "",
      "2021": "42"
    },
    "부상자": {
 